# MugheadWalker — Colab Intro

연세대학교 교양수업 **인공지능의 이해와 활용** RL 경쟁 프로젝트용 환경.
BipedalWalker를 포크해서 머그컵 모양의 상체가 payload 3개를 운반하도록 변형한 Gymnasium 환경.

이 노트북은:
1. Colab에서 `MugheadWalker-v0` 환경을 설치/사용하는 방법을 보여주고,
2. 랜덤 policy가 돌아가는 모습을 영상으로 확인하고,
3. PPO로 간단히 200K 스텝 학습하고,
4. 학습된 에이전트를 평가/시각화합니다.

> 💡 **GPU는 끄고** 돌리는 것을 권장합니다 (`런타임 → 런타임 유형 변경 → 하드웨어 가속기: CPU`). PPO MlpPolicy + Box2D는 CPU가 오히려 빠릅니다.


## 1. 설치

In [ ]:
# SWIG은 Box2D 빌드에 필요. mughead_walker는 github에서 바로 설치.
!pip install -q swig
!pip install -q git+https://github.com/essohn/mughead_walker.git
!pip install -q "stable-baselines3>=2.5" tensorboard imageio[ffmpeg]

## 2. 헤드리스 렌더링 설정

Colab에는 화면이 없어서 pygame의 `render_mode="human"` (실시간 창)은 안 됩니다.
대신 `rgb_array`로 프레임을 모아서 mp4로 저장한 뒤 재생합니다.


In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # pygame 헤드리스 모드

import numpy as np
import gymnasium as gym
import mughead_walker  # MugheadWalker-v0 등록

env = gym.make("MugheadWalker-v0")
obs, info = env.reset(seed=0)
print(f"observation shape: {obs.shape}")     # (42,)
print(f"action shape:      {env.action_space.shape}")  # (5,)
print(f"info keys:         {list(info.keys())}")
env.close()

### 환경 스펙 요약

| 항목 | 값 |
| --- | --- |
| Observation | 42-dim float32 (chassis pose/vel, 관절 4개, LIDAR 10, waist 각도/속도, payload 3개 × 5) |
| Action | 5-dim `[-1, 1]` (hip1, knee1, hip2, knee2, **waist**) |
| Reward | 전진 + payload 유지 보너스 − 낙오 페널티 − 넘어짐 −100 |
| 종료 | chassis 또는 mug가 지면에 닿으면 종료 |


## 3. 랜덤 에이전트 영상

In [ ]:
import imageio
from IPython.display import Video

env = gym.make("MugheadWalker-v0", render_mode="rgb_array")
obs, _ = env.reset(seed=0)

frames = []
for _ in range(400):
    frames.append(env.render())
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
env.close()

imageio.mimsave("random.mp4", frames, fps=30)
print(f"{len(frames)} frames 저장됨")
Video("random.mp4", embed=True, width=600)

## 4. PPO 학습 (200K 스텝, 약 2~4분)

실제 경쟁용 모델은 최소 1M 스텝을 권장하지만, 이 노트북은 동작 확인용으로 200K만 돌립니다.


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

def make_env(seed=0):
    def _init():
        env = gym.make("MugheadWalker-v0")
        env = Monitor(env)
        env.reset(seed=seed)
        return env
    return _init

# Colab 무료 티어에서는 DummyVecEnv(4~8)가 안정적.
vec = DummyVecEnv([make_env(i) for i in range(4)])

model = PPO(
    "MlpPolicy",
    vec,
    verbose=1,
    seed=0,
    device="cpu",
    tensorboard_log="./tb/",
)
model.learn(total_timesteps=200_000, progress_bar=True)
model.save("quick_ppo.zip")
print("학습 완료 — quick_ppo.zip 저장됨")

## 5. 평가 (5 에피소드, deterministic)

In [ ]:
env = gym.make("MugheadWalker-v0")

rewards, payloads, distances, successes = [], [], [], []
for ep in range(5):
    obs, _ = env.reset(seed=1000 + ep)
    total = 0.0
    info = {}
    terminated = truncated = False
    fell = False
    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total += float(reward)
        if terminated and reward <= -80:
            fell = True
    rewards.append(total)
    payloads.append(info.get("payloads_remaining", 0))
    distances.append(info.get("distance", 0.0))
    successes.append(0 if fell else 1)
    print(f"ep {ep}: reward={total:7.1f}  payloads={payloads[-1]}  distance={distances[-1]:5.1f}  success={bool(successes[-1])}")

env.close()

print()
print(f"mean reward:    {np.mean(rewards):7.1f}")
print(f"mean payloads:  {np.mean(payloads):.2f} / 3")
print(f"mean distance:  {np.mean(distances):.1f} m")
print(f"success rate:   {np.mean(successes)*100:.0f}%")

## 6. 학습된 에이전트 영상

In [ ]:
env = gym.make("MugheadWalker-v0", render_mode="rgb_array")
obs, _ = env.reset(seed=42)

frames = []
for _ in range(1600):
    frames.append(env.render())
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
env.close()

imageio.mimsave("trained.mp4", frames, fps=30)
print(f"{len(frames)} frames 저장됨")
Video("trained.mp4", embed=True, width=600)

## 7. TensorBoard (선택)

학습 중 reward/payload/fall 커브를 실시간으로 보고 싶다면:


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./tb/

## 8. 파라미터 튜닝

환경 난이도를 바꾸고 싶다면 `gym.make`에 kwargs로 전달하세요:

```python
env = gym.make(
    "MugheadWalker-v0",
    num_payloads=3,            # 0~3
    payload_mass_ratio=0.08,   # 0.06이 기본, 클수록 어려움
    payload_bounciness=0.25,   # 0.15 기본, 클수록 어려움
    mug_inner_width=45.0,      # 50 기본, 작을수록 어려움
)
```

자세한 설명과 round별 권장 파라미터는 [`docs/baseline/ppo_baseline_report.md`](https://github.com/essohn/mughead_walker/blob/main/docs/baseline/ppo_baseline_report.md) 참고.

## 9. 다음 단계

- **더 오래 학습하기**: `total_timesteps=1_000_000`으로 바꾸고 1M 스텝 (약 10~20분).
- **하이퍼파라미터 튜닝**: `learning_rate`, `n_steps`, `batch_size` 등을 `PPO(...)`에 넘길 수 있습니다. SB3 문서 참고.
- **경쟁 제출**: 저장한 `.zip` 모델을 로컬로 다운로드해서 `training/evaluate.py`(또는 `./scripts/play.sh`)로 벤치마크.
- **GitHub**: https://github.com/essohn/mughead_walker